In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "abfss://bronze@ecommercestoragev2.dfs.core.windows.net/merchants"
silver_path = "abfss://silver@ecommercestoragev2.dfs.core.windows.net/merchants"

hlp_path = f"{silver_path}/merchants_hlp"
dim_path = f"{silver_path}/merchants_dim"

batch_id = "0"

In [0]:
silver_df = (
    spark.read.format("parquet").load(bronze_path)
    .select("merchant_id", "store_name", "email")
    .distinct()
)

hlp_df = (
    spark.read.format("delta").load(hlp_path)
    .select("merchant_id", "merchant_sk")
)

In [0]:
source_df = (
    silver_df
    .join(hlp_df, on="merchant_id", how="left")
    .filter(F.col("merchant_sk").isNotNull())
    .select(
        F.col("merchant_sk"),
        F.col("store_name"),
        F.col("email"),
        F.current_timestamp().alias("created_at"),   # derived
        F.current_timestamp().alias("updated_at"),   # derived
        F.lit("merchants").alias("source"),
        F.lit("I").alias("oper"),
        F.current_timestamp().alias("ins_tmstmp"),
        F.current_timestamp().alias("upd_tmstmp"),
        F.lit(batch_id).alias("batch_id"),
    )
)

In [0]:
if not DeltaTable.isDeltaTable(spark, dim_path):
    # First run — write directly
    (
        source_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(dim_path)
    )
    print("Bootstrap: merchant_dim created.")

else:
    dim_table = DeltaTable.forPath(spark, dim_path)

    (
        dim_table.alias("target")
        .merge(
            source_df.alias("source"),
            "target.merchant_sk = source.merchant_sk"
        )
        .whenMatchedUpdate(set={
            "merchant_sk":  "source.merchant_sk",
            "store_name":   "source.store_name",
            "email":       "source.email",
            "updated_at":  "source.updated_at",
            "oper":        F.lit("U"),
            "upd_tmstmp":  "source.upd_tmstmp",
            "batch_id":    "source.batch_id",
        })
        .whenNotMatchedInsert(values={
            "merchant_sk":  "source.merchant_sk",
            "store_name":   "source.store_name",
            "email":        "source.email",
            "created_at":   "source.created_at",
            "updated_at":   "source.updated_at",
            "source":       "source.source",
            "oper":         "source.oper",
            "ins_tmstmp":   "source.ins_tmstmp",
            "upd_tmstmp":   "source.upd_tmstmp",
            "batch_id":     "source.batch_id",
        })
        .execute()
    )
    print("Incremental: customer_dim merged.")